# Week 1: reading a codebase like hepattn

This notebook is a tour of the Python features from this week's lesson, looked at *from the outside*
with `inspect`. You will use these same tools on hepattn itself: when a config says
`class_path: hepattn.models.Encoder`, `inspect.signature` tells you every option it accepts.

Run each cell with **Shift+Enter**. Cells marked **Your turn** are for you to edit.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

## 1. Signatures are the documentation (and the config schema)

`inspect.signature` shows a function's parameters, defaults and type hints. LightningCLI reads exactly
this to decide what a YAML block may contain (week 7).

In [ ]:
import inspect

from minihep.models.encoder import Encoder, EncoderLayer

print("Encoder", inspect.signature(Encoder.__init__))
print("EncoderLayer", inspect.signature(EncoderLayer.__init__))

Notice `**layer_kwargs` on `Encoder`: anything it does not recognise is forwarded to every `EncoderLayer`.
That is how `hybrid_norm: true` in a hepattn encoder config reaches the layers.

## 2. Registries: strings to classes

In [ ]:
from minihep.models.norm import NORM_TYPES

for name, cls in NORM_TYPES.items():
    print(f"{name:14s} -> {cls.__module__}.{cls.__qualname__}")

norm = NORM_TYPES["RMSNorm"](8)  # what Residual(..., norm="RMSNorm") does internally
print(norm)

## 3. Closures carry state

A closure stores the variables it captured in `__closure__`. This is how a FlexAttention `mask_mod`
remembers its window size.

In [ ]:
from exercises.week01.ex02_closures import make_window_mask, make_wrapped_window_mask

try:
    mask = make_window_mask(4)
    print("captured:", [c.cell_contents for c in mask.__closure__])
    print("mask(0, 2) =", mask(0, 2), "| mask(0, 3) =", mask(0, 3))
except NotImplementedError as err:
    print("Finish exercise 2 first:", err)

In [ ]:
# The late-binding trap, live
fs = [lambda x: i * x for i in range(4)]
print([f(10) for f in fs], "<- every lambda sees the final i")

## 4. The dict-key convention

Every tensor in hepattn travels in a flat dict. Keys are built with f-strings from an object name and a
field. Learn to read them as `{object}_{field}`.

In [ ]:
keys = ["hit_x", "hit_valid", "hit_embed", "particle_valid", "particle_hit_valid", "track_hit_logit", "key_is_hit"]
for key in keys:
    head, _, rest = key.partition("_")
    print(f"{key:20s} object={head:9s} field={rest}")

## Your turn

1. Clone hepattn on your laptop (week 1 lesson, section "Setting up") and find every registry:
   `git grep -n "_TYPES\b = \|_fns = {\|SOLVERS = {" src/hepattn`.
2. Run `inspect.signature` on `hepattn.models.MaskFormer.__init__` (inside `pixi shell` in the hepattn repo)
   and match each parameter to a key in `src/hepattn/experiments/trackml/configs/tracking-eta4-pt600-flex.yaml`.
3. Find the line in `hepattn/models/encoder.py` that writes `self.seq_len[0] = seq_len` and explain, in a
   comment cell below, why it cannot be `self.seq_len = torch.tensor([seq_len])`.